# Canadian Data Analyst Job Market Analysis (Job Bank postings, Jan 2025 - Aug 2026)

All the code we built so far, in the order it was run, with comments. Cells are grouped into parts:

0. Setup
1. Inspecting the raw files (encoding / separator / header bug)
2. The reusable loader `load_one_file()`
3. Combining and filtering all 20 months into `combined_full`
4. Task 2: postings trend (the original 11-month analysis, Oct 2025 - Aug 2026)
5. Salary analysis (11 months)
6. Does job-title mix explain the salary rise?
7. Extending to 20 months: schema check, volume vs. total postings, and the informatics-pool split

Note: parts 4 and 5 were first run on the 11-month data (Oct 2025 - Aug 2026). In this notebook they use `files_11` / `combined`, which are the same 11 months, so the numbers match what you saw.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Folder holding all 20 monthly Job Bank CSV exports (Jan 2025 - Aug 2026)
data_folder = Path('/Users/abinashkanagalingam/Downloads/DA_Job_Postings')

# glob('*.csv') finds every CSV in the folder; sorted() puts them in a stable order
files = sorted(data_folder.glob('*.csv'))
print(f"Total files found: {len(files)}")

# Quick look at each file: name, extension, and size in MB (a very different size can signal a different format)
for f in files:
    print(f"{f.name:20} {f.suffix:6} {f.stat().st_size / 1_000_000:.1f} MB")

## 1. Inspecting the raw files

The monthly exports are not all saved the same way. Some are UTF-8 with commas, some are UTF-16 with tabs, and the UTF-8 ones have a corrupted first column name (`ythWIC Job Location Snapshot ID`). These cells are how we found that out. They are diagnostic: you only need to re-run them if a new file misbehaves.

In [ ]:
# Step 1: read the first 20 raw bytes of each file (binary mode, 'rb') to spot the BOM signature.
#   b'\xef\xbb\xbf' at the start = UTF-8 with BOM
#   b'\xff\xfe'     at the start = UTF-16 (little endian)
for f in files:
    with open(f, 'rb') as fh:
        print(f.name, fh.read(20))

In [ ]:
# Step 2: try to read each file's first line as UTF-8.
# UTF-16 files fail here with UnicodeDecodeError (byte 0xff is never a valid UTF-8 start byte).
for f in files:
    try:
        with open(f, encoding='utf-8') as fh:
            print(f.name, '->', fh.readline()[:80])
    except UnicodeDecodeError as e:
        print(f.name, '-> FAILED:', e)

In [ ]:
# Step 3: same test as UTF-16. UTF-8 files fail here instead.
for f in files:
    try:
        with open(f, encoding='utf-16') as fh:
            print(f.name, '->', fh.readline()[:80])
    except UnicodeError as e:
        print(f.name, '-> FAILED:', e)

In [ ]:
# Step 4: record each file's encoding (decided by its first 2 bytes), then print the header line with repr().
# repr() shows hidden characters, so we can see whether columns are separated by \t (tab) or , (comma)
# and whether the first column name has the stray "yth" prefix.
encodings = {}
for f in files:
    with open(f, 'rb') as fh:
        first2 = fh.read(2)
    encodings[f.name] = 'utf-16' if first2 in (b'\xff\xfe', b'\xfe\xff') else 'utf-8-sig'

print(encodings)

for f in files:
    with open(f, encoding=encodings[f.name]) as fh:
        print(f.name, repr(fh.readline()[:100]))

In [ ]:
# Step 5: load one file of each type with its confirmed settings and compare.
#   UTF-16 files -> tab separated; UTF-8 files -> comma separated
utf16_sample = 'april_2026.csv'
utf8_sample = 'aug_2026.csv'

df_utf16 = pd.read_csv(data_folder / utf16_sample, encoding='utf-16', sep='\t')
df_utf8 = pd.read_csv(data_folder / utf8_sample, encoding='utf-8-sig', sep=',')

print(utf16_sample)
print(df_utf16.shape)
print(list(df_utf16.columns)[:5])
print()
print(utf8_sample)
print(df_utf8.shape)
print(list(df_utf8.columns)[:5])

In [ ]:
# Step 6: confirm the "yth" header bug appears in several UTF-8 files (nrows=1 reads just the header + 1 row, fast).
utf8_files = ['aug_2026.csv', 'jan_2026.csv', 'june_2026.csv']

for name in utf8_files:
    df_temp = pd.read_csv(data_folder / name, encoding='utf-8-sig', sep=',', nrows=1)
    print(name, '->', df_temp.columns[0])

## 2. The reusable loader

One function that handles both file types: detects the encoding from the first 2 bytes, picks the matching separator, and fixes the `yth` prefix if it is there.

In [ ]:
def load_one_file(filepath):
    # Detect encoding from the first 2 bytes (BOM signature)
    with open(filepath, 'rb') as fh:
        first2 = fh.read(2)
    encoding = 'utf-16' if first2 in (b'\xff\xfe', b'\xfe\xff') else 'utf-8-sig'

    # UTF-16 files are tab-separated, UTF-8 files are comma-separated
    sep = '\t' if encoding == 'utf-16' else ','

    # (Optional) add low_memory=False to silence the harmless "Columns (57,58) have mixed types" DtypeWarning
    df = pd.read_csv(filepath, encoding=encoding, sep=sep)

    # Fix the 'yth' prefix bug on the first column name, only if present
    first_col = df.columns[0]
    if first_col.startswith('yth'):
        df = df.rename(columns={first_col: first_col.replace('yth', '', 1)})

    return df

In [ ]:
# Test the loader on every file: each should load, and the first column should be clean ("WIC Job Location Snapshot ID").
# 2025 files have 64 columns, 2026 files have 65.
for f in files:
    df = load_one_file(f)
    print(f.name, df.shape, '-> first column:', df.columns[0])

## 3. Combine and filter all 20 months

"Data Analyst" is defined strictly as a title containing the text `data analyst` (case-insensitive). Adjacent roles (Business Analyst, BI Analyst, Data Scientist) are not mixed in.

In [ ]:
def get_month_label_v2(filename):
    # Every file is named monthname_year.csv (e.g. 'march_2026.csv' or 'jan_2025.csv').
    # Split on '_', keep the first 3 letters of the month, and attach the year -> 'Mar 2026'.
    month_part, year = filename.replace('.csv', '').split('_')
    return month_part[:3].title() + ' ' + year

all_filtered_full = []

for f in files:
    df = load_one_file(f)                                            # load one monthly file
    mask = df['Job Title'].str.contains('data analyst', case=False, na=False)   # True where the title contains "data analyst"
    filtered = df.loc[mask, ['Job Title', 'NOC21 Code Name', 'Province/Territory', 'City',
                              'First Posting Date', 'Vacancy Count']].copy()   # keep only matching rows and 6 columns
    filtered.insert(0, 'Month', get_month_label_v2(f.name))          # add a readable Month column at the front
    print(f.name, '->', get_month_label_v2(f.name), '->', len(filtered), 'Data Analyst rows')
    all_filtered_full.append(filtered)

# Stack the 20 small tables into one, and add a real date column so months sort chronologically
combined_full = pd.concat(all_filtered_full, ignore_index=True)
combined_full['Month_Date'] = pd.to_datetime(combined_full['Month'], format='%b %Y')
print('\nTotal combined rows:', len(combined_full))

In [ ]:
# Sanity checks: no 'Unknown' labels, every month present
print(combined_full['Month'].value_counts())
combined_full.head(3)

In [ ]:
# The original Task 2 and salary work used the 11 months Oct 2025 - Aug 2026.
# files_11 and combined recreate exactly that subset from the 20-month data.
files_11 = [f for f in files
            if pd.to_datetime(get_month_label_v2(f.name), format='%b %Y') >= pd.Timestamp('2025-10-01')]
combined = combined_full[combined_full['Month_Date'] >= '2025-10-01'].copy()

print(len(files_11), 'files,', len(combined), 'rows')   # expect 11 files, 240 rows

## 4. Task 2: postings trend over time (11 months, Oct 2025 - Aug 2026)

In [ ]:
# One row per month with the number of Data Analyst postings.
#   groupby('Month_Date').size()  -> counts rows per month (a Series with Month_Date as its index)
#   .reset_index(name=...)        -> turns it back into a normal table with Month_Date as a column
monthly_counts = combined.groupby('Month_Date').size().reset_index(name='Posting_Count')
monthly_counts = monthly_counts.sort_values('Month_Date')

# .dt.strftime('%b %Y') turns the real date back into readable text ('Mar 2026') for chart labels
monthly_counts['Month_Label'] = monthly_counts['Month_Date'].dt.strftime('%b %Y')
monthly_counts

In [ ]:
# Line chart of postings per month, with a callout on the March 2026 spike
plt.figure(figsize=(10, 5))
plt.plot(monthly_counts['Month_Label'], monthly_counts['Posting_Count'], marker='o')
plt.title('Data Analyst Job Postings by Month (Oct 2025 - Aug 2026)')
plt.xlabel('Month')
plt.ylabel('Number of Postings')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

# annotate(text, xy=point the arrow points to, xytext=where the text sits)
plt.annotate('Includes a multi-city\ngovernment hiring campaign\n(12 postings, same day)',
             xy=('Mar 2026', 38), xytext=('Mar 2026', 32),
             ha='center', fontsize=8, color='gray',
             arrowprops=dict(arrowstyle='->', color='gray'))

plt.tight_layout()
plt.show()

In [ ]:
# Put a number on the direction: fit a straight line through the points.
# Month_Index is a simple numeric time axis (0 = Oct 2025 ... 10 = Aug 2026) because polyfit needs numbers.
monthly_counts['Month_Index'] = range(len(monthly_counts))

# np.polyfit(x, y, 1) fits a degree-1 polynomial (a straight line) by least squares and returns slope and intercept
slope, intercept = np.polyfit(monthly_counts['Month_Index'], monthly_counts['Posting_Count'], 1)

print(f"Slope: {slope:.2f} postings per month")
print(f"Intercept: {intercept:.2f}")

In [ ]:
# Draw the fitted trend line over the actual data (y = slope * x + intercept for every month)
trend_line = slope * monthly_counts['Month_Index'] + intercept

plt.figure(figsize=(10, 5))
plt.plot(monthly_counts['Month_Label'], monthly_counts['Posting_Count'], marker='o', label='Actual postings')
plt.plot(monthly_counts['Month_Label'], trend_line, linestyle='--', color='red', label='Trend line')

plt.title('Data Analyst Job Postings by Month (Oct 2025 - Aug 2026)')
plt.xlabel('Month')
plt.ylabel('Number of Postings')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.legend()

plt.annotate('Includes a multi-city\ngovernment hiring campaign\n(12 postings, same day)',
             xy=('Mar 2026', 38), xytext=('Mar 2026', 32),
             ha='center', fontsize=8, color='gray',
             arrowprops=dict(arrowstyle='->', color='gray'))

plt.tight_layout()
plt.show()

In [ ]:
# Investigate the March 2026 spike (38 postings vs. 16-26 in other months): pull out just March's rows
march_data = combined[combined['Month'] == 'Mar 2026']

# Which titles make up March? (28 of 38 are 'data analyst - informatics and systems')
print(march_data['Job Title'].value_counts())
print()

# Which posting dates? (12 of them on 2026/03/25 alone -> a batch posting, not steady demand)
print(march_data['First Posting Date'].value_counts().sort_index())

In [ ]:
# Sensitivity check: does the March spike drive the downward slope?
# Copy first so the original table is untouched, then remove the 12 same-day campaign postings from March.
adjusted_counts = monthly_counts.copy()
adjusted_counts.loc[adjusted_counts['Month_Label'] == 'Mar 2026', 'Posting_Count'] -= 12

adj_slope, adj_intercept = np.polyfit(adjusted_counts['Month_Index'], adjusted_counts['Posting_Count'], 1)

print(f"Original slope:  {slope:.2f} postings/month")
print(f"Adjusted slope:  {adj_slope:.2f} postings/month (with campaign spike removed)")

# The slopes are identical because March is the middle month (index 5 of 0-10):
# a point at the center of the x-axis can move the line up or down, but cannot tilt it.

## 5. Salary analysis (11 months)

The raw data has structured salary columns: `Salary Per` (Hour / Year / Month / Bi-weekly), `Salary Minimum`, `Salary Maximum`. We average min and max per posting, then convert everything to an annual figure so rows are comparable.

In [ ]:
# Separate pass over the files (independent of `combined`), keeping the salary columns.
salary_rows = []

for f in files_11:
    df = load_one_file(f)
    mask = df['Job Title'].str.contains('data analyst', case=False, na=False)
    filtered = df.loc[mask, ['Job Title', 'First Posting Date', 'Salary Per', 'Salary Minimum', 'Salary Maximum']].copy()
    filtered.insert(0, 'Month', get_month_label_v2(f.name))
    salary_rows.append(filtered)

salary_data = pd.concat(salary_rows, ignore_index=True)
print('Total rows:', len(salary_data))
salary_data.head(10)

In [ ]:
# Per-posting midpoint of the pay range. Valid within a row regardless of unit.
salary_data['Salary_Avg'] = (salary_data['Salary Minimum'] + salary_data['Salary Maximum']) / 2

# Which pay units exist? (Hour, Year, Bi-weekly, Month -- plus a few blanks)
print(salary_data['Salary Per'].value_counts())

In [ ]:
# Convert every row to an annual figure so hourly, monthly, bi-weekly and yearly pay are comparable.
#   Hour      -> x 1950  (assumption: 37.5 hours/week x 52 weeks; state this in the write-up)
#   Month     -> x 12
#   Bi-weekly -> x 26   (26 pay periods a year)
#   Year      -> unchanged
def to_annual(row):
    if row['Salary Per'] == 'Hour':
        return row['Salary_Avg'] * 1950
    elif row['Salary Per'] == 'Month':
        return row['Salary_Avg'] * 12
    elif row['Salary Per'] == 'Bi-weekly':
        return row['Salary_Avg'] * 26
    elif row['Salary Per'] == 'Year':
        return row['Salary_Avg']
    else:
        return None      # missing / unrecognized unit

# apply(..., axis=1) runs the function on every row
salary_data['Salary_Annual'] = salary_data.apply(to_annual, axis=1)
salary_data[['Salary Per', 'Salary_Avg', 'Salary_Annual']].head(10)

In [ ]:
# How many rows have no usable salary? (4 expected: postings with no salary listed)
salary_data['Salary_Annual'].isna().sum()

In [ ]:
# First monthly salary table (this version still contains one bad row -- see the next cells)
salary_data['Month_Date'] = pd.to_datetime(salary_data['Month'], format='%b %Y')

# .agg() computes several things per group in one step.
# Posting_Count is the number of postings WITH a salary, so we can see how reliable each month's average is.
monthly_salary = salary_data.groupby('Month_Date').agg(
    Salary_Avg_Annual=('Salary_Annual', 'mean'),
    Posting_Count=('Salary_Annual', 'count')
).reset_index()

monthly_salary = monthly_salary.sort_values('Month_Date')
monthly_salary['Month_Label'] = monthly_salary['Month_Date'].dt.strftime('%b %Y')
monthly_salary

In [ ]:
# March 2026 averaged ~$218K, about 3x every other month. Look at the individual March salaries, highest first.
march_salary = salary_data[salary_data['Month'] == 'Mar 2026']
march_salary[['Job Title', 'Salary Per', 'Salary Minimum', 'Salary Maximum', 'Salary_Annual']].sort_values('Salary_Annual', ascending=False)

# One posting lists a maximum of $5,308.29 per HOUR (min $34.90) -> ~$5.2 million a year. That is a data entry error.

In [ ]:
# Confirm that one row explains the spike: March's average with and without it
march_salary_fixed = march_salary[march_salary['Salary_Annual'] < 1000000]

print("March average WITH the outlier:   ", march_salary['Salary_Annual'].mean())
print("March average WITHOUT the outlier:", march_salary_fixed['Salary_Annual'].mean())

In [ ]:
# Apply the same sanity cutoff ($1M a year) to the whole table.
# This also drops rows with missing salary (None < 1000000 is False): expect 240 -> 235 (4 missing + 1 outlier).
salary_data_clean = salary_data[salary_data['Salary_Annual'] < 1000000]
print(f"Rows before: {len(salary_data)}, Rows after removing outlier(s): {len(salary_data_clean)}")

monthly_salary = salary_data_clean.groupby('Month_Date').agg(
    Salary_Avg_Annual=('Salary_Annual', 'mean'),
    Posting_Count=('Salary_Annual', 'count')
).reset_index()

monthly_salary = monthly_salary.sort_values('Month_Date')
monthly_salary['Month_Label'] = monthly_salary['Month_Date'].dt.strftime('%b %Y')
monthly_salary

In [ ]:
# Salary trend: same straight-line fit as the postings trend, on the cleaned monthly averages
monthly_salary['Month_Index'] = range(len(monthly_salary))
salary_slope, salary_intercept = np.polyfit(monthly_salary['Month_Index'], monthly_salary['Salary_Avg_Annual'], 1)
salary_trend_line = salary_slope * monthly_salary['Month_Index'] + salary_intercept

plt.figure(figsize=(10, 5))
plt.plot(monthly_salary['Month_Label'], monthly_salary['Salary_Avg_Annual'], marker='o', color='green', label='Avg annual salary')
plt.plot(monthly_salary['Month_Label'], salary_trend_line, linestyle='--', color='red', label='Trend line')
plt.title('Average Annual Salary for Data Analyst Postings by Month (Outlier Corrected)')
plt.xlabel('Month')
plt.ylabel('Average Salary (Annualized, CAD)')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Salary slope: ${salary_slope:,.0f} per month")   # about +$199 per month

## 6. Does job-title mix explain the salary rise?

If higher-paying titles became a bigger share of postings later in the period, average pay could rise without any individual role paying more.

In [ ]:
# Average salary and number of postings per job title, highest paying first.
# Titles with count = 1 are single postings and not reliable averages.
salary_data_clean.groupby('Job Title')['Salary_Annual'].agg(['mean', 'count']).sort_values('mean', ascending=False)

In [ ]:
# Is there an Experience Level (seniority) column we could use instead? Look for related column names.
df_sample = load_one_file(data_folder / 'aug_2026.csv')
print([col for col in df_sample.columns if 'Experience' in col or 'Skill' in col or 'Education' in col])

In [ ]:
# 'Experience Level' exists. How complete is it for Data Analyst postings, file by file?
missing_counts = []

for f in files_11:
    df_temp = load_one_file(f)
    mask_temp = df_temp['Job Title'].str.contains('data analyst', case=False, na=False)
    total = mask_temp.sum()
    missing = df_temp.loc[mask_temp, 'Experience Level'].isna().sum()
    missing_counts.append({'File': f.name, 'Total_DA_Rows': total, 'Missing_Experience': missing})

pd.DataFrame(missing_counts)   # ~82% missing overall -> not usable, so we fall back to Job Title as the proxy

In [ ]:
# Count postings per month per job title.
#   groupby([...]).size()  -> counts for each month/title pair
#   .unstack(fill_value=0) -> months become rows, titles become columns, gaps filled with 0
#   .reindex(...)          -> put the rows back in chronological order
monthly_title_mix = salary_data_clean.groupby(['Month', 'Job Title']).size().unstack(fill_value=0)
monthly_title_mix = monthly_title_mix.reindex(monthly_salary['Month_Label'])
monthly_title_mix

In [ ]:
# Convert counts to percentage share per month for the 3 titles with real volume,
# so months with different totals can be compared fairly.
focus_titles = ['big data analyst', 'business data analyst', 'data analyst - informatics and systems']
mix_share = monthly_title_mix[focus_titles].div(monthly_title_mix[focus_titles].sum(axis=1), axis=0) * 100
mix_share.round(1)

# Finding: "business data analyst" (the highest-paying of the three) grew from roughly 5-12% of postings
# to 16-32% in the second half of the window -> part of the pay rise is a mix shift, not pure wage growth.
# Small counts, one job board, 11 months: treat as a soft signal, not a confirmed trend.

## 7. Extending to 20 months (Jan 2025 - Aug 2026)

In [ ]:
# 2025 files have 64 columns, 2026 files have 65. Which column differs?
# set(...) turns a column list into an unordered collection; set subtraction shows what is in one but not the other.
df_2025 = load_one_file(data_folder / 'jan_2025.csv')
df_2026 = load_one_file(data_folder / 'jan_2026.csv')

cols_2025 = set(df_2025.columns)
cols_2026 = set(df_2026.columns)

print("In 2026 but not 2025:", cols_2026 - cols_2025)   # only 'Official Language' -- not used in our analysis
print("In 2025 but not 2026:", cols_2025 - cols_2026)

In [ ]:
# Total postings in each file vs. Data Analyst postings, so we can tell a market-wide shrink from a Data-Analyst-specific one.
rows = []

for f in files:
    df = load_one_file(f)
    mask = df['Job Title'].str.contains('data analyst', case=False, na=False)
    rows.append({
        'Month': get_month_label_v2(f.name),
        'Total_Postings': len(df),        # all rows in the file
        'DA_Postings': mask.sum()         # rows whose title contains "data analyst"
    })

volume = pd.DataFrame(rows)
volume['Month_Date'] = pd.to_datetime(volume['Month'], format='%b %Y')
volume = volume.sort_values('Month_Date').reset_index(drop=True)
volume['DA_Share_Pct'] = (volume['DA_Postings'] / volume['Total_Postings'] * 100).round(3)   # DA postings as % of all postings
volume

In [ ]:
# Jan-Mar 2025 has 120-161 Data Analyst postings a month vs. ~20 later. Is one title responsible?
early = combined_full[combined_full['Month_Date'] <= '2025-03-01']
print(early['Job Title'].value_counts().head(10))     # 390 of 429 are 'data analyst - informatics and systems'
print()
print(early['First Posting Date'].value_counts().head(5))

In [ ]:
# Tag each row: True if the title is exactly the repeated pool title, False otherwise.
#   .str.lower() lowercases the title so capital letters cannot break the match
#   == '...' is an exact comparison (unlike .str.contains, which matches any title containing the text)
combined_full['Is_Informatics'] = combined_full['Job Title'].str.lower() == 'data analyst - informatics and systems'

# Count per month and per group; False (other titles) sorts before True (informatics), so name the columns in that order
monthly_split = combined_full.groupby(['Month_Date', 'Is_Informatics']).size().unstack(fill_value=0)
monthly_split.columns = ['Other_DA', 'Informatics_Systems']
monthly_split['Total'] = monthly_split.sum(axis=1)      # should match the Data Analyst counts per month
monthly_split

In [ ]:
# Stacked bar chart: how much of each month's total is the informatics pool vs. everything else.
# Run this cell once: it replaces the date index with text labels.
monthly_split.index = monthly_split.index.strftime('%b %Y')

monthly_split[['Other_DA', 'Informatics_Systems']].plot(kind='bar', stacked=True, figsize=(11, 5))
plt.title('Data Analyst Postings by Month: Informatics Pool vs. All Other Titles')
plt.xlabel('Month')
plt.ylabel('Number of Postings')
plt.xticks(rotation=45)
plt.legend(['Other DA titles', 'Informatics & systems'])
plt.tight_layout()
plt.show()

## Next steps

- Re-run the salary analysis (section 5) and title-mix check (section 6) on all 20 months.
- Forecast the next 4 months with a backtest (hold out the last 4 months, compare simple models).